# Hugging Face Open-Model Pulse

This notebook tracks real Hugging Face model metadata snapshots. A single snapshot is a cross-sectional table; repeated snapshots become a time series.

No synthetic fallback is used. If the Hub API is unavailable, the notebook stops.


In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd

# Prefer the checkout when this notebook is run inside the repository.
repo_root = Path.cwd()
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]:
    if (candidate / "src" / "detime").exists():
        repo_root = candidate
        break
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root))

from examples.hot_trends.data import (
    HotTrendDataError,
    append_real_snapshot,
    build_arxiv_monthly_counts,
    fetch_coingecko_market_chart,
    fetch_defillama_stablecoin_chains,
    fetch_github_repo_metadata,
    fetch_github_stargazers,
    fetch_huggingface_models,
    fetch_wikipedia_pageviews,
    source_audit_table,
)
from examples.hot_trends.decomposition import (
    component_summary,
    decompose_table,
    editorial_priority,
    residual_event_table,
)
from examples.hot_trends.scoring import article_language_guardrails

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)

CACHE_DIR = repo_root / "examples" / "hot_trends" / "cache"
OUTPUT_DIR = repo_root / "examples" / "hot_trends" / "outputs"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def save_table(df, name):
    path = OUTPUT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    print(f"saved: {path.relative_to(repo_root)}")
    return path


## 1. Fetch a real model snapshot


In [ ]:
snapshot = fetch_huggingface_models(limit=50, sort="downloads", direction=-1)
snapshot.head(20)


## 2. Audit the snapshot


In [ ]:
snapshot_audit = pd.DataFrame([{
    "snapshot_date": snapshot["snapshot_date"].iloc[0],
    "models": int(len(snapshot)),
    "non_null_downloads": int(snapshot["downloads"].notna().sum()),
    "non_null_likes": int(snapshot["likes"].notna().sum()),
    "source": "Hugging Face Hub API",
}])
snapshot_audit


## 3. Append snapshot to a real local log

The log is real only because each row comes from the API. It may contain just one snapshot on first run; collect daily or weekly to decompose adoption momentum.


In [ ]:
log_path = CACHE_DIR / "hf_model_snapshot_log.csv"
log = append_real_snapshot(snapshot, log_path)
log.tail(20)


## 4. Convert repeated snapshots to a time series if enough data exists


In [ ]:
log["snapshot_date"] = pd.to_datetime(log["snapshot_date"])
log["downloads"] = pd.to_numeric(log["downloads"], errors="coerce")
series_log = log.dropna(subset=["model_id", "downloads"]).sort_values(["model_id", "snapshot_date"])
# Hugging Face downloads are a snapshot metric rather than a guaranteed cumulative counter.
# Use the observed download level from repeated real snapshots instead of fabricating a delta.
ready_models = series_log.groupby("model_id")["snapshot_date"].nunique().loc[lambda s: s >= 4].index.tolist()
ready_models[:10], len(ready_models)


## 5. Decompose only if repeated real snapshots exist

This check prevents a fake time series. On first run, the notebook exports the snapshot and tells the reader to collect more real snapshots.


In [ ]:
if ready_models:
    decomp_input = series_log[series_log["model_id"].isin(ready_models)].rename(columns={"snapshot_date": "date", "model_id": "series", "downloads": "count"})
    decomp_input = decomp_input.dropna(subset=["count"])
    components = decompose_table(decomp_input, entity_col="series", time_col="date", value_col="count", method="MA_BASELINE", period=7, trend_window=3, transform="log1p")
    summary = editorial_priority(component_summary(components, entity_col="series", time_col="date"), entity_col="series")
    events = residual_event_table(components, entity_col="series", time_col="date", top_n=20)
else:
    components = pd.DataFrame()
    summary = pd.DataFrame([{"status": "not_enough_real_snapshots", "required": "collect at least 4 snapshot dates per model before decomposition"}])
    events = pd.DataFrame()
summary


## 6. Snapshot ranking for immediate publication

This is cross-sectional, not decomposed. It is still useful as a real source table.


In [ ]:
snapshot_rank = snapshot.sort_values(["downloads", "likes"], ascending=False, na_position="last").head(25)
snapshot_rank[["model_id", "pipeline_tag", "downloads", "likes", "last_modified", "source"]]


## 7. Guardrails


In [ ]:
guardrails = article_language_guardrails()
guardrails


In [ ]:
save_table(snapshot_audit, "03_hf_snapshot_audit")
save_table(snapshot_rank, "03_hf_snapshot_rank")
save_table(summary, "03_hf_decomposition_or_collection_status")
if not events.empty:
    save_table(events, "03_hf_residual_events")
save_table(guardrails, "03_hf_guardrails")
